In [1]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

In [2]:
import pathlib
import logging
from typing import Any

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.modules.loss import _Loss
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.utils.tensorboard.summary import hparams
from tqdm import tqdm

from src.dataset import SE3TransformerCP3DDataModule
from se3_transformer.se3_transformer.model import SE3TransformerPooled, Fiber
from se3_transformer.se3_transformer.runtime.callbacks import (
    AllMetricsCallback,
    CosineLRSchedulerCallback,
    BaseCallback,
    EarlyStoppingCallback,
)
from se3_transformer.se3_transformer.runtime.inference import evaluate
from se3_transformer.se3_transformer.runtime.loggers import LoggerCollection, Logger
from src.utils import (
    to_cuda,
    get_local_rank,
    rank_zero_only,
    using_tensor_cores,
    seed_everything,
)

torch.set_float32_matmul_precision("high")

/home/pham/miniforge3/envs/cp3d/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class ARGS:
    def __init__(self, configs):
        for key, value in configs.items():
            setattr(self, key, value)


configs = {
    "data_dir": pathlib.Path("pdb_data"),
    "seed": 43,
    "amp": True,
    "log_dir": pathlib.Path("results/se3t/cp3d"),
    "epochs": 10,
    "eval_interval": 2,
    "save_ckpt_path": pathlib.Path("results/se3t/cp3d/new_env_test.pth"),
    "learning_rate": 0.002,
    "min_learning_rate": 1e-7,
    "silent": False,
    "accumulate_grad_batches": 1,
    "batch_size": 64,
}

args = ARGS(configs)

In [4]:
def save_state(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    path: pathlib.Path,
    callbacks: dict[str, BaseCallback],
):
    """Saves model, optimizer and epoch states to path (only once per node)"""
    if get_local_rank() == 0:
        state_dict = model.state_dict()
        checkpoint = {
            "state_dict": state_dict,
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
        }
        for _, callback in callbacks.items():
            callback.on_checkpoint_save(checkpoint)

        torch.save(checkpoint, str(path))
        logging.info(f"Saved checkpoint to {str(path)}")


def load_state(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    path: pathlib.Path,
    callbacks: dict[str, BaseCallback],
):
    """Loads model, optimizer and epoch states from path"""
    checkpoint = torch.load(
        str(path), map_location={"cuda:0": f"cuda:{get_local_rank()}"}
    )
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    for _, callback in callbacks.items():
        callback.on_checkpoint_load(checkpoint)

    logging.info(f"Loaded checkpoint from {str(path)}")
    return checkpoint["epoch"]


def train_epoch(
    model,
    train_dataloader,
    loss_fn,
    epoch_idx,
    grad_scaler,
    optimizer,
    local_rank,
    callbacks,
    args,
):
    loss_acc = torch.zeros((1,), device="cuda")
    for i, batch in tqdm(
        enumerate(train_dataloader),
        total=len(train_dataloader),
        unit="batch",
        desc=f"Epoch {epoch_idx}",
        disable=local_rank != 0,
        # leave=False,
    ):
        *inputs, target = to_cuda(batch)

        for _, callback in callbacks.items():
            callback.on_batch_start()

        with torch.amp.autocast("cuda", enabled=args.amp):
            pred = model(*inputs)
            loss = loss_fn(pred.flatten(), target.flatten())

        loss_acc += loss.detach()
        grad_scaler.scale(loss).backward()

        # gradient accumulation
        if (i + 1) % 1 == 0 or (i + 1) == len(train_dataloader):
            grad_scaler.step(optimizer)
            grad_scaler.update()
            model.zero_grad(set_to_none=True)

    return loss_acc / (i + 1)


def train(
    model: nn.Module,
    loss_fn: _Loss,
    train_dataloader: DataLoader,
    val_dataloader: DataLoader,
    callbacks: dict[str, BaseCallback],
    logger: Logger,
    args,
):
    device = torch.cuda.current_device()
    model.to(device=device)
    local_rank = get_local_rank()

    model.train()
    grad_scaler = torch.amp.GradScaler("cuda", enabled=args.amp)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=args.learning_rate,
        fused=True,
    )

    epoch_start = 0

    for _, callback in callbacks.items():
        callback.on_fit_start(optimizer, args, epoch_start)

    for epoch_idx in range(epoch_start, args.epochs):
        loss = train_epoch(
            model,
            train_dataloader,
            loss_fn,
            epoch_idx,
            grad_scaler,
            optimizer,
            local_rank,
            callbacks,
            args,
        )

        loss = loss.item()
        logging.info(f"Train loss: {loss}")
        logger.log_metrics({"train loss": float(loss)}, epoch_idx)

        if epoch_idx + 1 == args.epochs:
            logger.log_metrics({"final train loss": float(loss)})

        for _, callback in callbacks.items():
            callback.on_epoch_end(loss)

        if (
            args.eval_interval > 0 and (epoch_idx + 1) % args.eval_interval == 0
        ) or epoch_idx + 1 == args.epochs:
            val_loss = evaluate(model, val_dataloader, loss_fn, callbacks, args)
            model.train()

            for _, callback in callbacks.items():
                callback.on_validation_end(epoch_idx, val_loss=val_loss)

        if callbacks["early_stopping"].early_stop:
            logging.info("Early stopping triggered")
            break

    if args.save_ckpt_path is not None:
        save_state(model, optimizer, args.epochs, args.save_ckpt_path, callbacks)

    for _, callback in callbacks.items():
        callback.on_fit_end()


def print_parameters_count(model):
    num_params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logging.info(f"Number of trainable parameters: {num_params_trainable}")

In [5]:
class TensorBoardLogger(Logger):
    def __init__(self, save_dir: pathlib.Path, version: int | None = None):
        super().__init__()
        self.base_dir = pathlib.Path(save_dir)
        self._version = version if version is not None else self._get_next_version()

        # Always compute the log_dir so it's available on all ranks
        self.log_dir = self.base_dir / f"version_{self._version}"

        # Only rank 0 creates dirs and the writer
        if not dist.is_initialized() or dist.get_rank() == 0:
            self.log_dir.mkdir(parents=True, exist_ok=True)
            self.experiment = SummaryWriter(log_dir=self.log_dir)

    def _get_next_version(self) -> int:
        # Scan base_dir for version_N directories and return next N (or 0 if none)
        if not self.base_dir.exists():
            return 0
        existing = []
        for d in self.base_dir.iterdir():
            if d.is_dir() and d.name.startswith("version_"):
                try:
                    n = int(d.name.split("_", 1)[1])
                    existing.append(n)
                except (IndexError, ValueError):
                    pass
        return 0 if not existing else max(existing) + 1

    @property
    def version(self) -> int:
        return int(self._version)

    @rank_zero_only
    def log_hyperparams(self, params: dict[str, Any]) -> None:
        params = self._sanitize_params(params)
        metrics = {"hp_metric": -1}
        exp, ssi, sei = hparams(params, metrics)
        writer = self.experiment._get_file_writer()
        writer.add_summary(exp, None)
        writer.add_summary(ssi, None)
        writer.add_summary(sei, None)

    @rank_zero_only
    def log_metrics(self, metrics: dict[str, float], step: int | None = None) -> None:
        for k, v in metrics.items():
            if isinstance(v, torch.Tensor):
                v = v.item()
            if isinstance(v, dict):
                self.experiment.add_scalars(k, v, step)
            else:
                try:
                    self.experiment.add_scalar(k, v, step)
                except Exception as ex:
                    raise ValueError(
                        f"\n you tried to log {v} which is currently not supported. Try a dict or a scalar/tensor."
                    ) from ex

In [6]:
loss_fn = nn.L1Loss()
datamodule = SE3TransformerCP3DDataModule(
    data_dir=args.data_dir / "Hexene",
    batch_size=args.batch_size,
    seed=args.seed,
)
model = SE3TransformerPooled(
    fiber_in=Fiber({0: datamodule.NODE_FEATURE_DIM}),
    fiber_out=Fiber({0: 4 * 32}),
    fiber_edge=Fiber({0: datamodule.EDGE_FEATURE_DIM}),
    num_degrees=4,
    num_channels=32,
    output_dim=1,
    tensor_cores=using_tensor_cores(args.amp),
    pooling="max",
    num_layers=7,
    num_heads=8,
    channels_div=2,
)
loggers = [TensorBoardLogger(save_dir=args.log_dir)]
logger = LoggerCollection(loggers)
logger.log_metrics({"Seed": args.seed})
callbacks = {
    "all_metric": AllMetricsCallback(
        logger, rescale_factor=datamodule.rescale_factor, prefix="validation"
    ),
    "scheduler": CosineLRSchedulerCallback(logger, epochs=args.epochs),
    "early_stopping": EarlyStoppingCallback(7, 0.005),
}

In [7]:
logging.info("====== SE(3)-Transformer ======")
logging.info("|      Training procedure     |")
logging.info("===============================")

if args.seed is not None:
    logging.info(f"Using seed {args.seed}")
    seed_everything(args.seed)

print_parameters_count(model)
logger.log_hyperparams(configs)
train(
    model,
    loss_fn,
    datamodule.train_dataloader(),
    datamodule.val_dataloader(),
    callbacks,
    logger,
    args,
)

Epoch 9: 100%|██████████| 65/65 [00:07<00:00,  8.46batch/s]
